In [2]:
import json
from pathlib import Path

SOURCE_FILE = Path("/content/cell_division_machine_readable.json")

print("Checking source file...")
print("File exists:", SOURCE_FILE.exists())

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        "cell_division_machine_readable.json was not found in /content/"
    )

with open(SOURCE_FILE, "r", encoding="utf-8") as f:
    curriculum_data = json.load(f)

print("JSON loaded successfully.")
print("Top-level type:", type(curriculum_data).__name__)
print("Top-level keys:")
for key in curriculum_data.keys():
    print(" -", key)

Checking source file...
File exists: True
JSON loaded successfully.
Top-level type: dict
Top-level keys:
 - dataset_metadata
 - source_units
 - unit_definition
 - semantic_chunking_reference


In [3]:
source_units = curriculum_data["source_units"]

print("Inspecting source_units...")
print("Type:", type(source_units).__name__)
print("Number of top-level items:", len(source_units))

print("\nFirst item:")
print(json.dumps(source_units[0], ensure_ascii=False, indent=2))

Inspecting source_units...
Type: list
Number of top-level items: 63

First item:
{
  "source_unit_id": "p23_u01",
  "grade": 8,
  "subject": "Science",
  "lesson": "Cell Division",
  "page": 23,
  "section": "Lesson Heading",
  "content_type": "lesson_heading",
  "content_ar": "الدرس (3): انقسام الخلايا"
}


In [4]:
# Cell 3 — Validate source units before chunk construction

unit_ids = [unit["source_unit_id"] for unit in source_units]
pages = [unit["page"] for unit in source_units]

print("SOURCE UNIT VALIDATION")
print("=" * 50)

print("Total source units:", len(source_units))
print("Unique source unit IDs:", len(set(unit_ids)))

# Check duplicates
duplicates = sorted({
    unit_id
    for unit_id in unit_ids
    if unit_ids.count(unit_id) > 1
})

print("Duplicate IDs:", duplicates if duplicates else "None")

# Page distribution
print("\nSOURCE UNITS BY PAGE")
print("-" * 50)

for page in sorted(set(pages)):
    page_units = [
        unit["source_unit_id"]
        for unit in source_units
        if unit["page"] == page
    ]

    print(
        f"Page {page}: "
        f"{len(page_units)} units | "
        f"{page_units[0]} → {page_units[-1]}"
    )

# First and last source unit
print("\nBOUNDARY CHECK")
print("-" * 50)
print("First source unit:", unit_ids[0])
print("Last source unit:", unit_ids[-1])

# Check required fields
required_fields = {
    "source_unit_id",
    "page",
    "section",
    "content_type",
    "content_ar"
}

missing_fields = []

for unit in source_units:
    missing = required_fields - set(unit.keys())

    if missing:
        missing_fields.append({
            "source_unit_id": unit.get("source_unit_id", "UNKNOWN"),
            "missing": sorted(missing)
        })

print("\nREQUIRED FIELD CHECK")
print("-" * 50)

if missing_fields:
    print("Units with missing required fields:")
    for item in missing_fields:
        print(item)
else:
    print("All source units contain the required fields.")

# Final validation status
print("\nVALIDATION STATUS")
print("=" * 50)

if (
    len(source_units) == len(set(unit_ids))
    and not missing_fields
):
    print("PASS — Source units are ready for chunk configuration mapping.")
else:
    print("FAIL — Resolve the issues above before continuing.")

SOURCE UNIT VALIDATION
Total source units: 63
Unique source unit IDs: 63
Duplicate IDs: None

SOURCE UNITS BY PAGE
--------------------------------------------------
Page 23: 5 units | p23_u01 → p23_u05
Page 24: 7 units | p24_u01 → p24_u07
Page 25: 9 units | p25_u01 → p25_u09
Page 26: 10 units | p26_u01 → p26_u10
Page 27: 10 units | p27_u01 → p27_u10
Page 28: 12 units | p28_u01 → p28_u12
Page 29: 10 units | p29_u01 → p29_u10

BOUNDARY CHECK
--------------------------------------------------
First source unit: p23_u01
Last source unit: p29_u10

REQUIRED FIELD CHECK
--------------------------------------------------
All source units contain the required fields.

VALIDATION STATUS
PASS — Source units are ready for chunk configuration mapping.


In [5]:
# Cell 4 — Display the complete source-unit map

print("COMPLETE SOURCE UNIT MAP")
print("=" * 100)

for unit in source_units:
    unit_id = unit["source_unit_id"]
    page = unit["page"]
    section = unit["section"]
    content_type = unit["content_type"]

    # Keep the preview readable on one line
    content_preview = " ".join(unit["content_ar"].split())

    if len(content_preview) > 140:
        content_preview = content_preview[:140] + "..."

    print(f"\n{unit_id}")
    print(f"  Page: {page}")
    print(f"  Section: {section}")
    print(f"  Type: {content_type}")
    print(f"  Content: {content_preview}")

COMPLETE SOURCE UNIT MAP

p23_u01
  Page: 23
  Section: Lesson Heading
  Type: lesson_heading
  Content: الدرس (3): انقسام الخلايا

p23_u02
  Page: 23
  Section: نشاط (1): كائنات حية متنوعة
  Type: activity
  Content: قامت هند بجولة في مزرعة جدها، ثم التقطت مجموعة من الصور لعدد من الكائنات الحية الموجودة فيها ثم بحثت وكتبت عدد الكروموسومات على كل منها، تأم...

p23_u03
  Page: 23
  Section: نشاط (1): كائنات حية متنوعة
  Type: visual_information
  Content: تُظهر الصور أعداد الكروموسومات للكائنات الآتية: البطاطا 48، الحصان 64، النملة 48، الإنسان 46، الشمبانزي 48، الحمار 62، الخيار 14، الدجاج 78.

p23_u04
  Page: 23
  Section: نشاط (1): كائنات حية متنوعة
  Type: question
  Content: هل يوجد علاقة بين حجم الكائن الحي وعدد الكروموسومات في خلايا جسمه؟

p23_u05
  Page: 23
  Section: نشاط (1): كائنات حية متنوعة
  Type: question
  Content: هل عدد الكروموسومات يحدد صفات الكائن الحي؟ وضح ذلك.

p24_u01
  Page: 24
  Section: نشاط (2): الخلايا تضاعف أعدادها
  Type: activity
  Content: يوجد في صفك نسخ 

In [6]:
# Cell 5 — Build and validate A/B/C semantic chunk configurations

from collections import Counter

# ---------------------------------------------------------
# 1. Index source units
# ---------------------------------------------------------

source_by_id = {
    unit["source_unit_id"]: unit
    for unit in source_units
}

all_source_ids = list(source_by_id.keys())

print("Indexed source units:", len(source_by_id))


# ---------------------------------------------------------
# 2. Define chunk mappings
# ---------------------------------------------------------

CONFIG_A = {
    "A01": ["p23_u01", "p23_u02", "p23_u03", "p23_u04", "p23_u05"],

    "A02": ["p24_u01", "p24_u02", "p24_u03", "p24_u04"],

    "A03": ["p24_u05", "p24_u06", "p24_u07"],

    "A04": [
        "p25_u01", "p25_u02", "p25_u03",
        "p25_u04", "p25_u05", "p25_u06", "p25_u07"
    ],

    "A05": [
        "p25_u08", "p25_u09", "p26_u01"
    ],

    "A06": [
        "p26_u02", "p26_u03"
    ],

    "A07": [
        "p26_u04"
    ],

    "A08": [
        "p26_u05", "p26_u06"
    ],

    "A09": [
        "p26_u07"
    ],

    "A10": [
        "p26_u08"
    ],

    "A11": [
        "p26_u09", "p26_u10"
    ],

    "A12": [
        "p27_u01", "p27_u02", "p27_u03", "p27_u04",
        "p27_u05", "p27_u06", "p27_u07"
    ],

    "A13": [
        "p27_u08", "p27_u09", "p27_u10"
    ],

    "A14": [
        "p28_u01", "p28_u02", "p28_u03",
        "p28_u04", "p28_u05", "p28_u06",
        "p28_u07", "p28_u08", "p28_u09"
    ],

    # Down syndrome contextual introduction
    "A15": [
        "p28_u10", "p28_u11"
    ],

    # Preserve the cross-page source continuation:
    # p28_u12 continues directly into p29_u01.
    "A16": [
        "p28_u12",
        "p29_u01", "p29_u02", "p29_u03",
        "p29_u04", "p29_u05", "p29_u06", "p29_u07"
    ],

    "A17": [
        "p29_u08", "p29_u09", "p29_u10"
    ],
}


CONFIG_B = {
    "B01": [
        "p23_u01", "p23_u02", "p23_u03", "p23_u04", "p23_u05"
    ],

    "B02": [
        "p24_u01", "p24_u02", "p24_u03", "p24_u04",
        "p24_u05", "p24_u06", "p24_u07"
    ],

    "B03": [
        "p25_u01", "p25_u02", "p25_u03",
        "p25_u04", "p25_u05", "p25_u06", "p25_u07"
    ],

    "B04": [
        "p25_u08", "p25_u09", "p26_u01"
    ],

    "B05": [
        "p26_u02", "p26_u03", "p26_u04"
    ],

    "B06": [
        "p26_u05", "p26_u06", "p26_u07",
        "p26_u08", "p26_u09", "p26_u10"
    ],

    "B07": [
        "p27_u01", "p27_u02", "p27_u03", "p27_u04",
        "p27_u05", "p27_u06", "p27_u07"
    ],

    "B08": [
        "p27_u08", "p27_u09", "p27_u10"
    ],

    "B09": [
        "p28_u01", "p28_u02", "p28_u03",
        "p28_u04", "p28_u05", "p28_u06",
        "p28_u07", "p28_u08", "p28_u09"
    ],

    "B10": [
        "p28_u10", "p28_u11"
    ],

    # p28_u12 and p29_u01 remain together because they are
    # a continuous sentence in the original curriculum source.
    "B11": [
        "p28_u12",
        "p29_u01", "p29_u02", "p29_u03", "p29_u04",
        "p29_u05", "p29_u06", "p29_u07", "p29_u08",
        "p29_u09", "p29_u10"
    ],
}


CONFIG_C = {
    "C01": [
        "p23_u01", "p23_u02", "p23_u03", "p23_u04", "p23_u05"
    ],

    "C02": [
        "p24_u01", "p24_u02", "p24_u03", "p24_u04",
        "p24_u05", "p24_u06", "p24_u07"
    ],

    "C03": [
        "p25_u01", "p25_u02", "p25_u03",
        "p25_u04", "p25_u05", "p25_u06", "p25_u07"
    ],

    "C04": [
        "p25_u08", "p25_u09", "p26_u01"
    ],

    "C05": [
        "p26_u02", "p26_u03", "p26_u04",
        "p26_u05", "p26_u06", "p26_u07",
        "p26_u08", "p26_u09", "p26_u10"
    ],

    "C06": [
        "p27_u01", "p27_u02", "p27_u03", "p27_u04",
        "p27_u05", "p27_u06", "p27_u07"
    ],

    "C07": [
        "p27_u08", "p27_u09", "p27_u10"
    ],

    "C08": [
        "p28_u01", "p28_u02", "p28_u03",
        "p28_u04", "p28_u05", "p28_u06",
        "p28_u07", "p28_u08", "p28_u09"
    ],

    "C09": [
        "p28_u10", "p28_u11", "p28_u12",
        "p29_u01", "p29_u02", "p29_u03", "p29_u04",
        "p29_u05", "p29_u06", "p29_u07", "p29_u08",
        "p29_u09", "p29_u10"
    ],
}


# ---------------------------------------------------------
# 3. Human-readable titles
# ---------------------------------------------------------

TITLES_A = {
    "A01": "Chromosome Numbers in Living Organisms",
    "A02": "Cell Division Activity and Purpose",
    "A03": "Chromatin and Preparation for Cell Division",
    "A04": "Duplicated Chromosome Structure",
    "A05": "Somatic and Reproductive Cells",
    "A06": "Interphase",
    "A07": "Purpose and Occurrence of Mitosis",
    "A08": "Mitosis Activity Introduction and Prophase",
    "A09": "Metaphase",
    "A10": "Anaphase",
    "A11": "Telophase and Cytokinesis",
    "A12": "Mitosis Questions",
    "A13": "Plant and Animal Cell Division",
    "A14": "Meiosis",
    "A15": "Down Syndrome Curriculum Context",
    "A16": "Chromosome Number Change and Down Syndrome",
    "A17": "Chromosome 21 and Meiosis Error",
}

TITLES_B = {
    "B01": "Chromosome Numbers in Living Organisms",
    "B02": "Cell Division and Preparation",
    "B03": "Duplicated Chromosome Structure",
    "B04": "Somatic and Reproductive Cells",
    "B05": "Interphase and Purpose of Mitosis",
    "B06": "Mitosis Stages",
    "B07": "Mitosis Questions",
    "B08": "Plant and Animal Cell Division",
    "B09": "Meiosis",
    "B10": "Down Syndrome Curriculum Context",
    "B11": "Chromosome Number Change, Down Syndrome and Chromosome 21",
}

TITLES_C = {
    "C01": "Chromosome Numbers in Living Organisms",
    "C02": "Cell Division and Preparation",
    "C03": "Duplicated Chromosome Structure",
    "C04": "Somatic and Reproductive Cells",
    "C05": "Mitosis: Preparation, Purpose and Stages",
    "C06": "Mitosis Questions",
    "C07": "Plant and Animal Cell Division",
    "C08": "Meiosis",
    "C09": "Chromosome Number Changes and Down Syndrome",
}


# ---------------------------------------------------------
# 4. Validation function
# ---------------------------------------------------------

def validate_configuration(name, mapping):

    mapped_ids = [
        unit_id
        for chunk_units in mapping.values()
        for unit_id in chunk_units
    ]

    counts = Counter(mapped_ids)

    duplicates = sorted([
        unit_id
        for unit_id, count in counts.items()
        if count > 1
    ])

    missing = sorted(
        set(all_source_ids) - set(mapped_ids)
    )

    unknown = sorted(
        set(mapped_ids) - set(all_source_ids)
    )

    print(f"\n{name}")
    print("=" * 60)
    print("Number of chunks:", len(mapping))
    print("Mapped source units:", len(mapped_ids))
    print("Unique mapped units:", len(set(mapped_ids)))
    print("Missing units:", missing if missing else "None")
    print("Duplicate units:", duplicates if duplicates else "None")
    print("Unknown units:", unknown if unknown else "None")

    passed = (
        len(mapped_ids) == len(all_source_ids)
        and len(set(mapped_ids)) == len(all_source_ids)
        and not missing
        and not duplicates
        and not unknown
    )

    print(
        "Coverage:",
        f"{len(set(mapped_ids) & set(all_source_ids))}/{len(all_source_ids)}"
    )

    print(
        "STATUS:",
        "PASS" if passed else "FAIL"
    )

    return passed


# ---------------------------------------------------------
# 5. Validate all configurations
# ---------------------------------------------------------

result_A = validate_configuration(
    "CONFIGURATION A — FINE-GRAINED",
    CONFIG_A
)

result_B = validate_configuration(
    "CONFIGURATION B — BASELINE",
    CONFIG_B
)

result_C = validate_configuration(
    "CONFIGURATION C — BROADER",
    CONFIG_C
)


# ---------------------------------------------------------
# 6. Final status
# ---------------------------------------------------------

print("\nFINAL CONFIGURATION VALIDATION")
print("=" * 60)

if result_A and result_B and result_C:
    print("PASS — A, B and C cover all source units exactly once.")
    print("A chunks:", len(CONFIG_A))
    print("B chunks:", len(CONFIG_B))
    print("C chunks:", len(CONFIG_C))
else:
    print("FAIL — Do not continue to chunk construction.")

Indexed source units: 63

CONFIGURATION A — FINE-GRAINED
Number of chunks: 17
Mapped source units: 63
Unique mapped units: 63
Missing units: None
Duplicate units: None
Unknown units: None
Coverage: 63/63
STATUS: PASS

CONFIGURATION B — BASELINE
Number of chunks: 11
Mapped source units: 63
Unique mapped units: 63
Missing units: None
Duplicate units: None
Unknown units: None
Coverage: 63/63
STATUS: PASS

CONFIGURATION C — BROADER
Number of chunks: 9
Mapped source units: 63
Unique mapped units: 63
Missing units: None
Duplicate units: None
Unknown units: None
Coverage: 63/63
STATUS: PASS

FINAL CONFIGURATION VALIDATION
PASS — A, B and C cover all source units exactly once.
A chunks: 17
B chunks: 11
C chunks: 9


In [7]:
# Cell 6 — Construct chunk texts and measure actual chunk lengths

import pandas as pd
import re

# ---------------------------------------------------------
# 1. Helper functions
# ---------------------------------------------------------

def build_chunk_text(unit_ids):
    """
    Combine source-unit content in original curriculum order.
    No paraphrasing, summarization, or external content is added.
    """
    texts = [
        source_by_id[unit_id]["content_ar"].strip()
        for unit_id in unit_ids
    ]

    return "\n".join(texts)


def count_words(text):
    """
    Simple whitespace-based word count.
    Used only as a descriptive pre-execution length measure.
    """
    return len(text.split())


def count_characters(text):
    """
    Character count excluding whitespace.
    """
    return len(re.sub(r"\s+", "", text))


def get_pages(unit_ids):
    """
    Return source pages represented in the chunk.
    """
    return sorted({
        source_by_id[unit_id]["page"]
        for unit_id in unit_ids
    })


# ---------------------------------------------------------
# 2. Build configuration records
# ---------------------------------------------------------

def build_configuration_records(
    config_name,
    mapping,
    titles
):
    records = []

    for chunk_id, unit_ids in mapping.items():

        text = build_chunk_text(unit_ids)

        records.append({
            "configuration": config_name,
            "chunk_id": chunk_id,
            "title": titles[chunk_id],
            "source_unit_count": len(unit_ids),
            "source_unit_ids": ", ".join(unit_ids),
            "source_pages": ", ".join(
                str(page) for page in get_pages(unit_ids)
            ),
            "word_count": count_words(text),
            "character_count": count_characters(text),
            "text": text
        })

    return records


records_A = build_configuration_records(
    "A_Fine",
    CONFIG_A,
    TITLES_A
)

records_B = build_configuration_records(
    "B_Baseline",
    CONFIG_B,
    TITLES_B
)

records_C = build_configuration_records(
    "C_Broad",
    CONFIG_C,
    TITLES_C
)

all_chunk_records = (
    records_A +
    records_B +
    records_C
)

chunks_df = pd.DataFrame(all_chunk_records)


# ---------------------------------------------------------
# 3. Display chunk-level measurements
# ---------------------------------------------------------

print("CHUNK LENGTH MEASUREMENTS")
print("=" * 90)

display(
    chunks_df[
        [
            "configuration",
            "chunk_id",
            "title",
            "source_unit_count",
            "source_pages",
            "word_count",
            "character_count"
        ]
    ]
)


# ---------------------------------------------------------
# 4. Configuration-level statistics
# ---------------------------------------------------------

summary_df = (
    chunks_df
    .groupby("configuration")
    .agg(
        chunk_count=("chunk_id", "count"),
        min_words=("word_count", "min"),
        max_words=("word_count", "max"),
        mean_words=("word_count", "mean"),
        median_words=("word_count", "median"),
        min_characters=("character_count", "min"),
        max_characters=("character_count", "max"),
        mean_characters=("character_count", "mean")
    )
    .reset_index()
)

summary_df["mean_words"] = (
    summary_df["mean_words"].round(2)
)

summary_df["median_words"] = (
    summary_df["median_words"].round(2)
)

summary_df["mean_characters"] = (
    summary_df["mean_characters"].round(2)
)


print("\nCONFIGURATION LENGTH SUMMARY")
print("=" * 90)

display(summary_df)


# ---------------------------------------------------------
# 5. Verify constructed chunk count
# ---------------------------------------------------------

print("\nCONSTRUCTION CHECK")
print("=" * 90)

print("A constructed chunks:", len(records_A))
print("B constructed chunks:", len(records_B))
print("C constructed chunks:", len(records_C))

assert len(records_A) == 17
assert len(records_B) == 11
assert len(records_C) == 9

print(
    "PASS — Chunk texts were constructed successfully "
    "from the validated source-unit mappings."
)


# ---------------------------------------------------------
# 6. Save pre-overlap chunk data
# ---------------------------------------------------------

chunks_df.to_csv(
    "/content/chunk_configurations_pre_overlap.csv",
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    "/content/chunk_length_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(
    "/content/chunk_configurations_pre_overlap.csv"
)
print(
    "/content/chunk_length_summary.csv"
)

CHUNK LENGTH MEASUREMENTS


,configuration,chunk_id,title,source_unit_count,source_pages,word_count,character_count
0,A_Fine,A01,Chromosome Numbers in Living Organisms,5,23,81,391
1,A_Fine,A02,Cell Division Activity and Purpose,4,24,69,309
2,A_Fine,A03,Chromatin and Preparation for Cell Division,3,24,78,409
3,A_Fine,A04,Duplicated Chromosome Structure,7,25,120,613
4,A_Fine,A05,Somatic and Reproductive Cells,3,"25, 26",61,373
5,A_Fine,A06,Interphase,2,26,25,136
6,A_Fine,A07,Purpose and Occurrence of Mitosis,1,26,21,129
7,A_Fine,A08,Mitosis Activity Introduction and Prophase,2,26,26,145
8,A_Fine,A09,Metaphase,1,26,17,105
9,A_Fine,A10,Anaphase,1,26,12,69



CONFIGURATION LENGTH SUMMARY


,configuration,chunk_count,min_words,max_words,mean_words,median_words,min_characters,max_characters,mean_characters
0,A_Fine,17,12,132,61.24,59.0,69,659,319.71
1,B_Baseline,11,46,201,94.64,81.0,254,1007,494.09
2,C_Broad,9,55,253,115.67,120.0,305,1261,603.89



CONSTRUCTION CHECK
A constructed chunks: 17
B constructed chunks: 11
C constructed chunks: 9
PASS — Chunk texts were constructed successfully from the validated source-unit mappings.

Saved:
/content/chunk_configurations_pre_overlap.csv
/content/chunk_length_summary.csv


In [8]:
# Cell 7 — Construct frozen O0 and O1 overlap conditions

def build_overlap_records(
    config_name,
    mapping,
    titles,
    overlap_setting
):
    """
    Construct experimental chunks using the frozen overlap protocol.

    O0:
        No overlap.

    O1:
        For every chunk after the first, duplicate the final
        source unit of the immediately preceding semantic chunk
        at the beginning of the current retrieval representation.
    """

    chunk_items = list(mapping.items())
    records = []

    for index, (chunk_id, core_unit_ids) in enumerate(chunk_items):

        core_unit_ids = list(core_unit_ids)

        if overlap_setting == "O0":
            retrieval_unit_ids = list(core_unit_ids)
            overlap_unit_ids = []

        elif overlap_setting == "O1":

            if index == 0:
                overlap_unit_ids = []
            else:
                previous_core_units = chunk_items[index - 1][1]
                overlap_unit_ids = [previous_core_units[-1]]

            retrieval_unit_ids = (
                overlap_unit_ids +
                core_unit_ids
            )

        else:
            raise ValueError(
                f"Unknown overlap setting: {overlap_setting}"
            )

        retrieval_text = build_chunk_text(
            retrieval_unit_ids
        )

        core_text = build_chunk_text(
            core_unit_ids
        )

        records.append({
            "configuration": config_name,
            "overlap_setting": overlap_setting,
            "chunk_id": chunk_id,
            "title": titles[chunk_id],

            # Original semantic identity
            "core_source_unit_ids": ", ".join(core_unit_ids),

            # Context duplicated by O1
            "overlap_source_unit_ids": ", ".join(overlap_unit_ids),

            # Actual representation that will be embedded
            "retrieval_source_unit_ids": ", ".join(retrieval_unit_ids),

            "core_word_count": count_words(core_text),
            "retrieval_word_count": count_words(retrieval_text),

            "core_text": core_text,
            "retrieval_text": retrieval_text
        })

    return records

In [9]:
experimental_records = []

configurations = [
    ("A_Fine", CONFIG_A, TITLES_A),
    ("B_Baseline", CONFIG_B, TITLES_B),
    ("C_Broad", CONFIG_C, TITLES_C)
]

for config_name, mapping, titles in configurations:

    experimental_records.extend(
        build_overlap_records(
            config_name,
            mapping,
            titles,
            "O0"
        )
    )

    experimental_records.extend(
        build_overlap_records(
            config_name,
            mapping,
            titles,
            "O1"
        )
    )

experimental_chunks_df = pd.DataFrame(
    experimental_records
)

print("EXPERIMENTAL CHUNK REPRESENTATIONS")
print("=" * 90)

display(
    experimental_chunks_df[
        [
            "configuration",
            "overlap_setting",
            "chunk_id",
            "overlap_source_unit_ids",
            "core_word_count",
            "retrieval_word_count"
        ]
    ]
)

EXPERIMENTAL CHUNK REPRESENTATIONS


,configuration,overlap_setting,chunk_id,overlap_source_unit_ids,core_word_count,retrieval_word_count
0,A_Fine,O0,A01,,81,81
1,A_Fine,O0,A02,,69,69
2,A_Fine,O0,A03,,78,78
3,A_Fine,O0,A04,,120,120
4,A_Fine,O0,A05,,61,61
...,...,...,...,...,...,...
69,C_Broad,O1,C05,p26_u01,133,155
70,C_Broad,O1,C06,p26_u10,59,75
71,C_Broad,O1,C07,p27_u07,55,61
72,C_Broad,O1,C08,p27_u10,132,146


In [10]:
# ---------------------------------------------------------
# Validation
# ---------------------------------------------------------

print("\nOVERLAP VALIDATION")
print("=" * 90)

for config_name, mapping, titles in configurations:

    subset_o0 = experimental_chunks_df[
        (experimental_chunks_df["configuration"] == config_name) &
        (experimental_chunks_df["overlap_setting"] == "O0")
    ]

    subset_o1 = experimental_chunks_df[
        (experimental_chunks_df["configuration"] == config_name) &
        (experimental_chunks_df["overlap_setting"] == "O1")
    ]

    expected_chunks = len(mapping)

    assert len(subset_o0) == expected_chunks
    assert len(subset_o1) == expected_chunks

    # O0 must contain no overlap units
    assert all(
        subset_o0["overlap_source_unit_ids"] == ""
    )

    # First O1 chunk must contain no previous context
    first_o1 = subset_o1.iloc[0]

    assert first_o1["overlap_source_unit_ids"] == ""

    # Every later O1 chunk must contain exactly
    # the final core source unit from the previous chunk.
    mapping_items = list(mapping.items())

    for i in range(1, len(mapping_items)):

        previous_chunk_units = mapping_items[i - 1][1]
        expected_overlap_unit = previous_chunk_units[-1]

        actual_overlap_unit = (
            subset_o1.iloc[i]["overlap_source_unit_ids"]
        )

        assert actual_overlap_unit == expected_overlap_unit

    print(
        f"{config_name}: PASS — "
        f"O0={len(subset_o0)} chunks, "
        f"O1={len(subset_o1)} chunks"
    )


print("\nFINAL OVERLAP STATUS")
print("=" * 90)
print(
    "PASS — Frozen O0 and O1 representations "
    "were constructed successfully."
)


# Save the exact representations that will later be embedded

experimental_chunks_df.to_csv(
    "/content/experimental_chunk_representations.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nSaved:")
print(
    "/content/experimental_chunk_representations.csv"
)


OVERLAP VALIDATION
A_Fine: PASS — O0=17 chunks, O1=17 chunks
B_Baseline: PASS — O0=11 chunks, O1=11 chunks
C_Broad: PASS — O0=9 chunks, O1=9 chunks

FINAL OVERLAP STATUS
PASS — Frozen O0 and O1 representations were constructed successfully.

Saved:
/content/experimental_chunk_representations.csv
